# QuickDraw CNN Colab Runner

This notebook sets up the project, downloads the CNN dataset, runs a smoke test, and gives a few experiment commands for the full CNN training workflow.

## 1. Clone the repo

Replace the repo URL below if needed.

In [4]:
!git clone https://github.com/simongydvandy/CS-4262-Doodling-Project.git
%cd CS-4262-Doodling-Project

Cloning into 'CS-4262-Doodling-Project'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 84 (delta 42), reused 47 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 148.07 KiB | 5.29 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/CS-4262-Doodling-Project


## 2. Install dependencies

Colab usually includes PyTorch already, but this ensures the remaining project packages are available.

In [5]:
!python -m pip install -q gdown numpy matplotlib scikit-learn requests

## 3. Verify runtime

If you want GPU acceleration, switch Colab to a GPU runtime before running training.

In [6]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

torch 2.10.0+cu128
cuda available: True
device: Tesla T4


## 4. Download the CNN data

In [7]:
!python scripts/download_quickdraw_data.py --role cnn


Files: ['X_cnn.npy', 'y_cnn.npy']

  ↓ Downloading X_cnn.npy...
Downloading...
From (original): https://drive.google.com/uc?id=1HfFQNydRtS7-AURmX_L01Y_lfsVmHSxm
From (redirected): https://drive.google.com/uc?id=1HfFQNydRtS7-AURmX_L01Y_lfsVmHSxm&confirm=t&uuid=94fcd430-ba25-4f95-bab1-50b556f9dd31
To: /content/CS-4262-Doodling-Project/data/processed/X_cnn.npy
100% 541M/541M [00:07<00:00, 75.2MB/s]
  ↓ Downloading y_cnn.npy...
Downloading...
From: https://drive.google.com/uc?id=1IEqzY6WYceUNzIcrfYT058Fs1ibNPx-h
To: /content/CS-4262-Doodling-Project/data/processed/y_cnn.npy
100% 5.52M/5.52M [00:00<00:00, 162MB/s]

Done! Files saved to data/processed/


## 5. Inspect the data shape

The QuickDraw CNN images are stored as flattened vectors of length 784. The training script reshapes them automatically.

In [8]:
import numpy as np
X = np.load('data/processed/X_cnn.npy', mmap_mode='r')
y = np.load('data/processed/y_cnn.npy', mmap_mode='r')
print('X shape:', X.shape, X.dtype)
print('y shape:', y.shape, y.dtype)
print('labels:', int(y.min()), 'to', int(y.max()))

X shape: (690000, 784) uint8
y shape: (690000,) int64
labels: 0 to 344


## 6. Run a smoke test

In [9]:
!python scripts/train_sketch_cnn.py --categories 10 --limit 10000 --epochs 5 --batch-size 256 --num-workers 0

Epoch 01/5 train_loss=1.8893 train_acc=0.3363 val_loss=1.4240 val_acc=0.5400 val_macro_f1=0.5148
Epoch 02/5 train_loss=1.3256 train_acc=0.5657 val_loss=1.0989 val_acc=0.6670 val_macro_f1=0.6658
Epoch 03/5 train_loss=1.1087 train_acc=0.6469 val_loss=1.0251 val_acc=0.6750 val_macro_f1=0.6699
Epoch 04/5 train_loss=0.9582 train_acc=0.7017 val_loss=0.8388 val_acc=0.7540 val_macro_f1=0.7512
Epoch 05/5 train_loss=0.8518 train_acc=0.7399 val_loss=0.7811 val_acc=0.7630 val_macro_f1=0.7608
Best epoch: 5
Test accuracy: 0.7615
Test macro-F1: 0.7642
Results written to: results


## 7. Run a full baseline experiment

In [10]:
!python scripts/train_sketch_cnn.py --epochs 20 --batch-size 256 --learning-rate 5e-4 --hidden-dim 512 --conv-channels 128 256 --num-workers 2

Epoch 01/15 train_loss=3.8670 train_acc=0.1954 val_loss=2.9260 val_acc=0.3692 val_macro_f1=0.3446
Epoch 02/15 train_loss=3.2904 train_acc=0.2813 val_loss=2.7061 val_acc=0.4062 val_macro_f1=0.3907
Epoch 03/15 train_loss=3.1393 train_acc=0.3082 val_loss=2.5924 val_acc=0.4265 val_macro_f1=0.4091
Epoch 04/15 train_loss=3.0290 train_acc=0.3274 val_loss=2.4860 val_acc=0.4520 val_macro_f1=0.4369
Epoch 05/15 train_loss=2.9028 train_acc=0.3524 val_loss=2.3701 val_acc=0.4669 val_macro_f1=0.4529
Epoch 06/15 train_loss=2.7867 train_acc=0.3737 val_loss=2.2954 val_acc=0.4832 val_macro_f1=0.4716
Epoch 07/15 train_loss=2.6865 train_acc=0.3946 val_loss=2.2229 val_acc=0.4988 val_macro_f1=0.4876
Epoch 08/15 train_loss=2.5974 train_acc=0.4116 val_loss=2.1495 val_acc=0.5110 val_macro_f1=0.5007
Epoch 09/15 train_loss=2.5201 train_acc=0.4271 val_loss=2.1029 val_acc=0.5204 val_macro_f1=0.5126
Epoch 10/15 train_loss=2.4435 train_acc=0.4426 val_loss=2.0691 val_acc=0.5279 val_macro_f1=0.5188
Epoch 11/15 train_lo

## 8. Try a few variants

Uncomment one at a time.

In [11]:
# Best shallow-model runs
# !python scripts/train_sketch_cnn.py --epochs 25 --batch-size 256 --learning-rate 5e-4 --hidden-dim 256 --conv-channels 64 128 --num-workers 2
# !python scripts/train_sketch_cnn.py --epochs 20 --batch-size 256 --learning-rate 5e-4 --hidden-dim 384 --conv-channels 96 192 --num-workers 2
# !python scripts/train_sketch_cnn.py --epochs 20 --batch-size 256 --learning-rate 5e-4 --hidden-dim 512 --conv-channels 128 256 --num-workers 2

# Best deep-model runs
# !python scripts/train_sketch_cnn.py --model-type deep --epochs 25 --batch-size 256 --learning-rate 5e-4 --hidden-dim 512 --conv-channels 64 128 256 --dropout 0.35 --num-workers 2
# !python scripts/train_sketch_cnn.py --model-type deep --epochs 25 --batch-size 256 --learning-rate 5e-4 --hidden-dim 512 --conv-channels 96 192 320 --dropout 0.35 --num-workers 2

# Best current ResNet run
# !python scripts/train_sketch_cnn.py --model-type resnet --epochs 25 --batch-size 256 --learning-rate 5e-4 --hidden-dim 512 --conv-channels 64 128 256 --dropout 0.35 --num-workers 2

# Best next push: ResNet + augmentation + scheduler
# !python scripts/train_sketch_cnn.py --model-type resnet --augment --scheduler plateau --epochs 30 --batch-size 256 --learning-rate 5e-4 --hidden-dim 512 --conv-channels 64 128 256 --dropout 0.35 --num-workers 2
# !python scripts/train_sketch_cnn.py --model-type resnet --augment --scheduler plateau --epochs 30 --batch-size 256 --learning-rate 5e-4 --hidden-dim 512 --conv-channels 96 192 320 --dropout 0.35 --num-workers 2
# !python scripts/train_sketch_cnn.py --model-type resnet --augment --scheduler cosine --epochs 30 --batch-size 128 --learning-rate 5e-4 --hidden-dim 512 --conv-channels 96 192 320 --dropout 0.35 --num-workers 2

Epoch 01/20 train_loss=3.0450 train_acc=0.3423 val_loss=2.2362 val_acc=0.4921 val_macro_f1=0.4831
Epoch 02/20 train_loss=2.3099 train_acc=0.4723 val_loss=1.9947 val_acc=0.5411 val_macro_f1=0.5339
Epoch 03/20 train_loss=2.1118 train_acc=0.5119 val_loss=1.8805 val_acc=0.5648 val_macro_f1=0.5592
Epoch 04/20 train_loss=1.9873 train_acc=0.5359 val_loss=1.7996 val_acc=0.5833 val_macro_f1=0.5788
Epoch 05/20 train_loss=1.8970 train_acc=0.5539 val_loss=1.7381 val_acc=0.5942 val_macro_f1=0.5897
Epoch 06/20 train_loss=1.8224 train_acc=0.5681 val_loss=1.7163 val_acc=0.5972 val_macro_f1=0.5919
Epoch 07/20 train_loss=1.7609 train_acc=0.5810 val_loss=1.6765 val_acc=0.6069 val_macro_f1=0.6032
Epoch 08/20 train_loss=1.7046 train_acc=0.5913 val_loss=1.6532 val_acc=0.6112 val_macro_f1=0.6074
Epoch 09/20 train_loss=1.6579 train_acc=0.6004 val_loss=1.6364 val_acc=0.6161 val_macro_f1=0.6126
Epoch 10/20 train_loss=1.6150 train_acc=0.6087 val_loss=1.6318 val_acc=0.6157 val_macro_f1=0.6126
Epoch 11/20 train_lo

## 9. Inspect outputs

In [12]:
!ls -lah results

total 28M
drwxr-xr-x 2 root root 4.0K Apr 10 23:28 .
drwxr-xr-x 8 root root 4.0K Apr 10 23:28 ..
-rw-r--r-- 1 root root  27M Apr 10 23:50 cnn_best.pt
-rw-r--r-- 1 root root  47K Apr 10 23:54 cnn_classification_report.json
-rw-r--r-- 1 root root 931K Apr 10 23:54 cnn_confusion_matrix.npy
-rw-r--r-- 1 root root 156K Apr 10 23:54 cnn_confusion_matrix.png
-rw-r--r-- 1 root root 5.5K Apr 10 23:54 cnn_history.json
-rw-r--r-- 1 root root  345 Apr 10 23:54 cnn_metrics.json
-rw-r--r-- 1 root root 101K Apr 10 23:54 cnn_training_curves.png


In [13]:
import json
from pathlib import Path

metrics_path = Path('results/cnn_metrics.json')
summary_path = Path('results/cnn_presentation_summary.json')
if not metrics_path.exists():
    raise FileNotFoundError('Run a training cell first, or restore results into results/.')

with metrics_path.open() as f:
    metrics = json.load(f)

headline = {
    'model_type': metrics.get('model_type'),
    'best_epoch': metrics.get('best_epoch'),
    'best_val_accuracy': metrics.get('best_val_accuracy'),
    'best_val_top_3_accuracy': metrics.get('best_val_top_3_accuracy'),
    'best_val_top_5_accuracy': metrics.get('best_val_top_5_accuracy'),
    'best_val_macro_f1': metrics.get('best_val_macro_f1'),
    'test_accuracy': metrics.get('test_accuracy'),
    'test_top_3_accuracy': metrics.get('test_top_3_accuracy'),
    'test_top_5_accuracy': metrics.get('test_top_5_accuracy'),
    'test_macro_f1': metrics.get('test_macro_f1'),
}
print(json.dumps(headline, indent=2))

if summary_path.exists():
    with summary_path.open() as f:
        summary = json.load(f)
    print('\nTop confusion pairs:')
    for row in summary.get('top_confusions', [])[:5]:
        print(f"- {row['true_label']} -> {row['predicted_label']}: {row['count']}")
    print('\nWeakest classes by F1:')
    for row in summary.get('worst_classes_by_f1', [])[:5]:
        print(f"- {row['label']}: f1={row['f1_score']:.3f}, recall={row['recall']:.3f}")

{'model': 'quickdraw_cnn',
 'device': 'cuda',
 'num_classes': 345,
 'num_train_samples': 483000,
 'num_val_samples': 69000,
 'num_test_samples': 138000,
 'best_epoch': 16,
 'best_val_accuracy': 0.6278695652173913,
 'best_val_macro_f1': 0.6245563172317992,
 'test_accuracy': 0.6277898550724638,
 'test_macro_f1': 0.6248190784250395}

## 10. Download the best artifacts

Run this after a good experiment to pull the outputs back to your machine.

In [14]:
from google.colab import files

for path in [
    'results/cnn_metrics.json',
    'results/cnn_presentation_summary.json',
    'results/cnn_classification_report.json',
    'results/cnn_history.json',
    'results/cnn_confusion_matrix.png',
    'results/cnn_training_curves.png',
    'results/cnn_best.pt',
]:
    files.download(path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>